# JEPA Weight Sweep - Deep Validation

**Why**: JEPA gives +9pp on visual tasks at w=0.1. Higher weights unstable.

**Goal**: Fine weight sweep (0.05/0.1/0.2) x 3 seeds x 2 visual tasks.

**Hardware**: 1 machine x 8 GPUs (~6h).

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Part A - Prior Results (st04 jepa_w0.1)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    sub = df_prior[(df_prior.stage == 'st04') & (df_prior.sweep == 'jepa_w0.1')]
    print(summary_stats(sub))
else:
    print('Prior data not found.')

In [ ]:
if df_prior is not None:
    plot_prior_bar(df_prior, ['cifar10','mazes'],
                   'st04', 'jepa_w0.1', 'Prior: JEPA w=0.1 vs baseline',
                   'figures/02_prior_bar.png')

In [ ]:
curves = load_prior_curves()
if curves:
    plot_prior_curves(curves, 'cifar10',
        [('st00','paper','baseline','#888'),
         ('st04','jepa_w0.1','JEPA 0.1','#1f77b4')],
        'cifar10 convergence (prior)', 'figures/02_prior_conv.png')

## Part B - Experiment Design (3 weights x 3 seeds x 2 tasks = 18 runs)

In [ ]:
exps = make_jepa(['cifar10','mazes'], [0,1,2], weights=[0.05, 0.1, 0.2])
print(f'{len(exps)} experiments')
for e in exps[:6]:
    print(f'  {e.name}')
print('  ...')

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/02_jepa', dry_run=True)

## Part C - Run Training

Set `CONFIRM_RUN = True` to launch (~6h).

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/02_jepa')


In [ ]:
status('logs/deep/02_jepa')

## Part D - Results Analysis

In [ ]:
df = collect('logs/deep/02_jepa')
if df.empty:
    print('No results yet.')
else:
    print(df[['name','task','best_acc','delta']].to_string(index=False))
    plot_delta_bars(df, 'JEPA sweep vs baseline', 'figures/02_delta.png')

In [ ]:
if not df.empty:
    import re
    df['weight'] = df['name'].str.extract(r'w([0-9]+p?[0-9]*)')[0].str.replace('p','.').astype(float)
    plot_sweep_curve(df, 'weight', title='JEPA weight sweep (errorbar = std over seeds)',
                    savepath='figures/02_sweep.png')

In [ ]:
if not df.empty:
    plot_box_seeds(df, 'task', 'best_acc',
                   'Seed variance per task', 'figures/02_box.png')
    print(summary_stats(df, groupby=('task','weight')))